## ML_1032: 2D Convolution

**Difficulty**: Medium | **TorchCode**: #22

## Problem

Implement a **2D convolution** from scratch with support for:

- **stride**
- **padding**

Use:

- `unfold` to extract sliding local patches
- `einsum` to apply the kernel to each patch

---

## Convolution Formula

For each output element:

$$
O_{b,o,i,j}
=
\sum_{c,p,q}
X_{b,c,\;s_h i + p,\;s_w j + q}\cdot W_{o,c,p,q} + b_o
$$

Where:

- \(b\): batch index
- \(o\): output channel index
- \(c\): input channel index
- \(i, j\): output spatial position
- \(p, q\): kernel position
- \(s_h, s_w\): stride along height and width
- \(X\): input tensor
- \(W\): convolution kernel
- \(b_o\): bias of output channel \(o\)

---

## Output Size Formula

Assume:

- input height: \(H\)
- input width: \(W\)
- kernel height: \(k_h\)
- kernel width: \(k_w\)
- padding: \((p_h, p_w)\)
- stride: \((s_h, s_w)\)

Then the output height and width are:

$$
H_{\text{out}} = \left\lfloor \frac{H + 2p_h - k_h}{s_h} \right\rfloor + 1
$$

$$
W_{\text{out}} = \left\lfloor \frac{W + 2p_w - k_w}{s_w} \right\rfloor + 1
$$

---

## Example

### Given

- Input: \(64 \times 32\)
  - \(H = 64\)
  - \(W = 32\)

- Kernel: \(3 \times 3\)
  - \(k_h = 3\)
  - \(k_w = 3\)

- Padding: \(1\)
  - \(p_h = 1\)
  - \(p_w = 1\)

- Stride: \(2\)
  - \(s_h = 2\)
  - \(s_w = 2\)

---

### Compute Output Height

$$
H_{\text{out}}
=
\left\lfloor
\frac{64 + 2\cdot 1 - 3}{2}
\right\rfloor + 1
$$

$$
=
\left\lfloor
\frac{63}{2}
\right\rfloor + 1
$$

$$
= 31 + 1 = 32
$$

---

### Compute Output Width

$$
W_{\text{out}}
=
\left\lfloor
\frac{32 + 2\cdot 1 - 3}{2}
\right\rfloor + 1
$$

$$
=
\left\lfloor
\frac{31}{2}
\right\rfloor + 1
$$

$$
= 15 + 1 = 16
$$

---

## Final Output Size

$$
32 \times 16
$$

So the output spatial size is:

- height: \(32\)
- width: \(16\)

---

## Intuition

- Padding \(=1\) adds a 1-pixel zero border around the input
- Kernel \(=3\times 3\) looks at one local \(3\times 3\) patch at a time
- Stride \(=2\) means the kernel moves 2 steps each time
- Because of stride 2, the output becomes smaller than the input
- `unfold` extracts all local patches
- `einsum` multiplies each patch with the kernel and sums over channels and kernel positions


In [ ]:
import torch
import torch.nn.functional as F

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    if padding > 0:
        x = F.pad(x, [padding] * 4)
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1
    patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
    out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
    if bias is not None:
        out = out + bias.view(1, -1, 1, 1)
    return out

In [ ]:
import torch
import torch.nn.functional as F

# --- Test 1: Output shape ---
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
assert my_conv2d(x, w).shape == (1, 16, 6, 6)

# --- Test 2: Matches F.conv2d ---
torch.manual_seed(0)
x = torch.randn(2, 3, 8, 8)
w = torch.randn(4, 3, 3, 3)
b = torch.randn(4)
out = my_conv2d(x, w, b)
ref = F.conv2d(x, w, b)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 3: With padding ---
torch.manual_seed(0)
x = torch.randn(1, 1, 5, 5)
w = torch.randn(1, 1, 3, 3)
out = my_conv2d(x, w, padding=1)
ref = F.conv2d(x, w, padding=1)
assert out.shape == ref.shape and torch.allclose(out, ref, atol=1e-4)

# --- Test 4: With stride ---
torch.manual_seed(0)
x = torch.randn(1, 1, 8, 8)
w = torch.randn(1, 1, 3, 3)
out = my_conv2d(x, w, stride=2)
ref = F.conv2d(x, w, stride=2)
assert out.shape == ref.shape and torch.allclose(out, ref, atol=1e-4)

# --- Test 5: Gradient flow ---
x = torch.randn(1, 1, 4, 4, requires_grad=True)
w = torch.randn(2, 1, 3, 3, requires_grad=True)
my_conv2d(x, w).sum().backward()
assert x.grad is not None and w.grad is not None

print('All tests passed!')